# Connect an agent to external services with MCP

This lab uses a real HTTPS MCP server hosted by your instructor. Its records and task/calendar destinations are fictional. Each participant has a separate sandbox. No laptop installation is required; Colab installs the client in its temporary runtime.

In Colab Secrets (key icon), add `COURSE_API_KEY`, `COURSE_MCP_TOKEN` and `COURSE_MCP_REVIEW_TOKEN`, using the values supplied by the instructor. Enable Notebook access. For the read-only demonstration also add `COURSE_MCP_READ_TOKEN`. Never paste keys into a prompt, notebook cell, Markdown file or shared screenshot.

The Qwen key pays for model calls. The MCP key permits access to the external service. The separate review key permits the human approval control. These are different responsibilities.


In [ ]:
#@title Open the course workspace
import os, sys, subprocess, json, urllib.request, importlib.util
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai==2.41.1', 'ipywidgets>=8,<9'], check=True)
BASE_URL = 'https://raw.githubusercontent.com/FeikoWielsma/Using-AI-Agents/main/'
def course_file(name):
    local_root = os.environ.get('COURSE_LOCAL_ROOT')
    if local_root:
        return Path(local_root) / name
    path = Path('course-assets') / name
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BASE_URL + name, path)
    return path
KEY = os.environ.get('COURSE_API_KEY')
if not KEY:
    try:
        from google.colab import userdata
        KEY = userdata.get('COURSE_API_KEY')
    except Exception:
        KEY = None
from openai import OpenAI
MODEL = 'qwen3.8-max'
EXTRA = {'enable_thinking': False}
client = OpenAI(api_key=KEY or 'course-key-not-configured',
    base_url='https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1',
    timeout=35.0, max_retries=0)
spec=importlib.util.spec_from_file_location('course_workbench',course_file('notebooks/course_workbench.py'))
workbench=importlib.util.module_from_spec(spec)
spec.loader.exec_module(workbench)
print('Course interface ready.' if KEY else 'Add COURSE_API_KEY in Colab Secrets, enable notebook access, and rerun this setup. Source browsing remains available.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mcp==2.1.1'], check=True)


## Six short demonstrations

1. **Discover capabilities.** Connect, then inspect tools. Read a tool description and its required inputs. The host discovers these from the server; the model does not magically know the service.
2. **Retrieve service information.** Select a case, list its documents, and ask the agent to read one. Inspect the tool name, arguments and returned records in the trace.
3. **Separate proposal from action.** Ask the agent to propose one follow-up. Review changes, then try Apply before approval. The server refuses; no task/event has been saved.
4. **Review and verify.** Read the complete displayed proposal, approve it, apply it and verify the destination. Apply again: it returns the same saved record rather than creating a duplicate.
5. **Restrict access.** Reconnect using the read-only key and request another proposal. Retrieval still works; the server refuses writes. Existing records remain.
6. **Disconnect.** Disconnect, then try reading or running the agent. Reconnect and verify the records persist. Disconnecting removes access from this notebook; it does not revoke the credential itself.

Run the cell below to open the controls. Allow 30–90 seconds for an agent request. Each tool result is visible in the trace.


In [ ]:
spec=importlib.util.spec_from_file_location('mcp_workbench',course_file('notebooks/mcp_workbench.py'))
mcp_ui=importlib.util.module_from_spec(spec)
spec.loader.exec_module(mcp_ui)
connection=mcp_ui.show(client)


## Module 2 activity

Allow about ten minutes. Use the browser lab's Connected services activity, or the Colab controls above.

1. Connect with read-and-propose access and select Fitness. Ask for a free twenty-minute planning-review slot on Thursday 10 September 2026, after the fictional business trip.
2. Inspect the proposed time and title. Approve and apply it, or reject it and ask for a different slot.
3. Retrieve the calendar and locate the saved appointment. Disconnect when finished.

The diary and calendar are fictional. The task is scheduling a review; no Garmin connection or workout prescription is involved. An existing appointment can affect a repeated run.

If time permits, select Sales for an account follow-up or Programme for a dependency task. Save connection notes only if useful; no procedure or memory document is required.

If the service is unavailable, the trainer can show a prepared connection walkthrough, labelled as a recording or simulation. The other browser activities run independently.
